# Fase 2 - Parte 1: Extração de Sintomas e Diagnóstico Assistido

Notebook da atividade `Diagnóstico Automatizado - IA no Estetoscópio Digital`.

Objetivo:
- ler frases simuladas de pacientes;
- identificar sintomas por regras simples;
- sugerir um possível diagnóstico com base em um mapa de conhecimento.

In [ ]:
from collections import Counter
from pathlib import Path
import re
import unicodedata

import pandas as pd

In [ ]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
FRASES_PATH = BASE_DIR / 'data' / 'frases_sintomas_pacientes.txt'
MAPA_PATH = BASE_DIR / 'data' / 'mapa_conhecimento_sintomas_doencas.csv'

with open(FRASES_PATH, encoding='utf-8') as arquivo:
    frases = [linha.strip() for linha in arquivo if linha.strip()]

mapa = pd.read_csv(MAPA_PATH)

print(f'Total de frases: {len(frases)}')
mapa.head()

In [ ]:
def normalizar_texto(texto: str) -> str:
    texto_sem_acentos = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    texto_limpo = re.sub(r'[^a-zA-Z0-9\s]', ' ', texto_sem_acentos.lower())
    return re.sub(r'\s+', ' ', texto_limpo).strip()


def identificar_diagnosticos(frase: str, mapa_conhecimento: pd.DataFrame) -> dict:
    frase_normalizada = normalizar_texto(frase)
    correspondencias = []

    for linha in mapa_conhecimento.itertuples(index=False):
        sintomas = []
        for coluna in ('sintoma_1', 'sintoma_2'):
            valor = getattr(linha, coluna)
            if isinstance(valor, str) and valor.strip():
                sintomas.append(normalizar_texto(valor))

        sintomas_encontrados = [s for s in sintomas if s and s in frase_normalizada]
        if sintomas_encontrados:
            correspondencias.append((linha.doenca_associada, sintomas_encontrados))

    if not correspondencias:
        return {
            'frase': frase,
            'sintomas_identificados': 'nenhum sintoma mapeado',
            'diagnostico_sugerido': 'Encaminhar para avaliacao clinica',
            'confianca_regra': 0,
        }

    contagem = Counter(item[0] for item in correspondencias)
    diagnostico_sugerido, confianca = contagem.most_common(1)[0]
    sintomas_unicos = sorted({sintoma for _, lista in correspondencias for sintoma in lista})

    return {
        'frase': frase,
        'sintomas_identificados': ', '.join(sintomas_unicos),
        'diagnostico_sugerido': diagnostico_sugerido,
        'confianca_regra': confianca,
    }

In [ ]:
resultados = [identificar_diagnosticos(frase, mapa) for frase in frases]
df_resultados = pd.DataFrame(resultados)
df_resultados

In [ ]:
df_resultados['diagnostico_sugerido'].value_counts()

## Interpretação

A lógica aplicada aqui é baseada em regras simples de correspondência textual.

Pontos fortes:
- fácil de explicar na apresentação;
- transparente para fins acadêmicos;
- adequada para demonstrar uma ontologia inicial.

Limitações:
- não entende contexto clínico complexo;
- depende muito das palavras cadastradas no CSV;
- não substitui avaliação médica.